#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Rate Distribution

In [ ]:
# Constrain By Zipcode
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    def clean_zip(val):
        val = str(val).strip()
        if '-' in val:
            val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        return val[:5]

    df['clean_zip'] = df['zip'].apply(clean_zip)
    df['zip_num'] = pd.to_numeric(df['clean_zip'], errors='coerce')

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        z = int(z)

        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'

        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'

        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'

        return 'Unknown'

    df['location_name'] = df['zip_num'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()

    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'

    df_filtered['State'] = df_filtered['location_name'].apply(get_state)

    print(f"Records remaining after ZIP filtering: {len(df_filtered)}")

    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    def get_location_type(loc):
        if str(loc).endswith('_S'): return 'Small Town/Rural'
        if str(loc).endswith('_M'): return 'Mid-Size City'
        return 'Urban/Metro'

    df_filtered['location_type'] = df_filtered['location_name'].apply(get_location_type)

    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25

        return pd.Series([std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['std', 'p10', 'p25', 'iqr']

    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 4, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 4, 1, 0)

    categories = sorted(df_filtered['category'].unique())
    target_states = ['NY', 'CA', 'GA']

    for category in categories:
        print(f"\nProcessing Category: {category}")

        for state in target_states:
            plot_data = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            if plot_data.empty:
                continue

            current_order = state_orders[state]
            current_order = [city for city in current_order if city in plot_data['location_name'].unique()]

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            sns.boxplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].set_xlabel('Region (SSMLL Order)')
            axes[0].set_ylabel('Rating Value')
            axes[0].tick_params(axis='x', rotation=45)

            sns.boxplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].set_xlabel('Region (SSMLL Order)')
            axes[1].set_ylabel('Std Dev')
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Boxplot) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            sns.violinplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].tick_params(axis='x', rotation=45)

            sns.violinplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Violin) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

    print("\nGenerating Histograms")
    for category in categories:
        for state in target_states:
            current_order = state_orders[state]
            for location in current_order:
                plot_df = df_filtered[
                    (df_filtered['category'] == category) &
                    (df_filtered['location_name'] == location)
                ]
                if plot_df.empty: continue

                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, ax=axes[0], color='blue')
                axes[0].set_title(f'Average Rating', fontsize=11)
                axes[0].set_xlabel('Rating Value')

                sns.histplot(data=plot_df, x='std', bins=15, kde=True, ax=axes[1], color='orange')
                axes[1].set_title(f'Standard Deviation', fontsize=11)
                axes[1].set_xlabel('Std Dev')

                fig.suptitle(get_fig_title(f'{state} - {location}: Histograms for {category}'), fontsize=14)
                plt.tight_layout()
                plt.show()

    print("\nLow Rating Analysis")

    low_avg_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'])
    rename_dict = {0: 'High Rating (>4)', 1: 'Low Rating (<=4)'}
    low_avg_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'], normalize='index') * 100

    print("Counts:")
    print(low_avg_crosstab.rename(columns=rename_dict))
    print("Percentages: Average Rating")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_avg_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title(get_fig_title('Count of Low Ratings (Avg <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='Avg Rating <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    low_p25_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'])
    low_p25_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'], normalize='index') * 100

    print("Counts:")
    print(low_p25_crosstab.rename(columns=rename_dict))
    print("Percentages:")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_p25_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title(get_fig_title('Count of Low Ratings (P25 <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='25th Percentile <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    print("\nScatter Plots (Split by State)")

    for state in target_states:
        cities_in_state = state_orders[state]
        cities_in_state = [c for c in cities_in_state if c in df_filtered['location_name'].unique()]

        if not cities_in_state: continue

        n_cities = len(cities_in_state)
        cols = 3
        rows = math.ceil(n_cities / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
        if n_cities > 1:
            axes = axes.flatten()
        else:
            axes = [axes]

        for i, city in enumerate(cities_in_state):
            subset = df_filtered[df_filtered['location_name'] == city]

            axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='25th Percentile')
            axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='10th Percentile')

            axes[i].set_title(f"{city} (n={len(subset)})")
            axes[i].grid(True, linestyle='--', alpha=0.5)
            if i == 0:
                axes[i].legend()

        for i, ax in enumerate(axes):
            if i >= (rows - 1) * cols:
                ax.set_xlabel("Average Rating")
            if i % cols == 0:
                ax.set_ylabel("Percentile Rating")
            ax.set_xlim(0.5, 5.5)
            ax.set_ylim(0.5, 5.5)

        if n_cities > 1:
            for j in range(n_cities, len(axes)):
                fig.delaxes(axes[j])

        fig.suptitle(get_fig_title(f"{state}: Scatter Plot (Avg vs P25/P10)"), fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    for category in categories:
        for state in target_states:
            category_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            cities_in_state = state_orders[state]
            cities_in_state = [c for c in cities_in_state if c in category_df['location_name'].unique()]

            if not cities_in_state: continue

            n_cities = len(cities_in_state)
            cols = 3
            rows = math.ceil(n_cities / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
            if n_cities > 1:
                axes = axes.flatten()
            else:
                axes = [axes]

            for i, city in enumerate(cities_in_state):
                subset = category_df[category_df['location_name'] == city]

                axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='P25')
                axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='P10')

                axes[i].set_title(f"{city} (n={len(subset)})")
                axes[i].grid(True, linestyle='--', alpha=0.5)
                if i == 0:
                    axes[i].legend()

            for i, ax in enumerate(axes):
                if i >= (rows - 1) * cols:
                    ax.set_xlabel("Avg Rating")
                if i % cols == 0:
                    ax.set_ylabel("Percentile")
                ax.set_xlim(0.5, 5.5)
                ax.set_ylim(0.5, 5.5)

            if n_cities > 1:
                for j in range(n_cities, len(axes)):
                    fig.delaxes(axes[j])

            fig.suptitle(get_fig_title(f"{state}: Scatter Plot for {category}"), fontsize=16)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

else:
    print("DataFrame is empty.")

# One Plot only avg

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

print(f"Loading data from: {final_csv_path}")

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    # Preprocess
    if 'search_location' not in df.columns:
        if 'location_name' in df.columns:
            df['search_location'] = df['location_name']
        else:
            print("Error: Missing location columns.")
            df = pd.DataFrame()

if not df.empty:
    # Define mapping
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }
    all_target_cities = [city for cities in state_orders.values() for city in cities]

    # Clean & Filter
    df['search_location'] = df['search_location'].astype(str)
    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()
    df_filtered['location_name'] = df_filtered['search_location']

    # Convert ratings
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    # Process stars
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    # Define location size type
    def get_location_info(loc):
        s_loc = str(loc)
        if s_loc.endswith('_S'):
            return 'Small'
        elif s_loc.endswith('_M'):
            return 'Mid-Size'
        elif s_loc.endswith('_L'):
            return 'Large'
        return 'Unknown'

    df_filtered['Location_Type'] = df_filtered['location_name'].apply(get_location_info)

    # Sort by population
    master_locations_order = [
        # Small
        'SaranacLake_NY_S',
        'FortBragg_CA_S',
        'Toccoa_GA_S',
        'Vidalia_GA_S',
        'Malone_NY_S',
        'Eureka_CA_S',

        # Mid-Size
        'Macon_GA_M',
        'Syracuse_NY_M',
        'Modesto_CA_M',

        # Large
        'Augusta_GA_L',
        'Buffalo_NY_L',
        'Atlanta_GA_L',
        'SanFrancisco_CA_L',
        'LA_CA_L',
        'NYC_NY_L'
    ]

    print("Sorted City Order:")
    print(master_locations_order)

    # Calculate Metrics
    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan])

        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25
        return pd.Series([p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['p10', 'p25', 'iqr']
    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low rating
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 1, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 1, 1, 0)

    categories = sorted(df_filtered['category'].unique())

    # Small, mid, large: blue, orange, green
    custom_colors = ['tab:blue', 'tab:orange', 'tab:green']

    # plot
    for category in categories:
        print(f"Category: {category}")
        cat_data = df_filtered[df_filtered['category'] == category]
        if cat_data.empty: continue

        current_order = [city for city in master_locations_order if city in cat_data['location_name'].unique()]

        # 1. Boxplots
        fig, ax = plt.subplots(figsize=(12, 8))
        hue_order = ['Small', 'Mid-Size', 'Large']

        sns.boxplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Average Rating by City (Sorted by Population)', fontsize=14)
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel("City (Small -> Large)")

        plt.tight_layout()
        plt.show()

        # 2. Violin
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.violinplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Avg Rating Distribution', fontsize=14)
        ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()


        # 3. Hist
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.histplot(data=subset, x='rating_value', bins=10, kde=False, stat='count',
                             color=custom_colors[i], ax=ax)
                ax.set_title(f'{loc_type} - Frequency')
                ax.set_xlabel('Rating')
                ax.set_ylabel('Count')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Rating Frequency', y=1.05)
        plt.tight_layout()
        plt.show()

        # 4. Normalized Hist
        print(f"[{category}] Histograms - Normalized")
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.histplot(data=subset, x='rating_value', bins=10, kde=False, stat='probability',
                             color=custom_colors[i], ax=ax)
                ax.set_title(f'{loc_type} - Normalized')
                ax.set_xlabel('Rating')
                ax.set_ylabel('Normalized Count')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Normalized Distribution', y=1.05)
        plt.tight_layout()
        plt.show()

        # 5. Scatter Plots
        print(f"[{category}] Scatter Plots (Avg vs P25)")
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.scatterplot(data=subset, x='rating_value', y='p25',
                                color=custom_colors[i], s=80, alpha=0.6, ax=ax)
                ax.set_title(f'{loc_type}')
                ax.set_xlabel('Average Rating')
                if i == 0:
                    ax.set_ylabel('25th Percentile')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Scatter: Avg vs P25', y=1.05)
        plt.tight_layout()
        plt.show()

    # Low rating analysis
    print("Low Rating Analysis")

    hue_order_bar = ['Small', 'Mid-Size', 'Large']
    rename_dict = {0: 'High (>1)', 1: 'Low (<=1)'}

    # Avg <= 1
    low_avg_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (Avg Rating <= 1)")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Low Rating (Avg <= 1) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

    # P25 <= 3
    low_p25_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (25th Percentile <= 1)")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title('Low Rating (P25 <= 1) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("DataFrame is empty.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

print(f"Loading data from: {final_csv_path}")

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    # Clean zip
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val:
            val = val.split('-')[0]
        # Keep only digits, and 5 first digit
        val = ''.join(filter(str.isdigit, val))

        val = val[:5]

        if val:
            return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        print("Warning: 'zip' column not found. Skipping ZIP filtering.")
        df['zip_clean'] = np.nan

    # zip mapping
    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try:
            z = int(z)
        except ValueError:
            return 'Unknown'

        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'

        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'

        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'

        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)

    print(f"Records before filtering: {len(df)}")

    # Clean up Unknowns
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Update location columns
    df['search_location'] = df['mapped_location']
    df['location_name'] = df['mapped_location']

    print(f"Records remaining after zipcode filtering: {len(df)}")


    # Define mapping
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }
    all_target_cities = [city for cities in state_orders.values() for city in cities]

    # Clean & Filter
    df['search_location'] = df['search_location'].astype(str)
    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()
    df_filtered['location_name'] = df_filtered['search_location']

    # Convert ratings
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    # Process stars
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    # Define location size type
    def get_location_info(loc):
        s_loc = str(loc)
        if s_loc.endswith('_S'):
            return 'Small'
        elif s_loc.endswith('_M'):
            return 'Mid-Size'
        elif s_loc.endswith('_L'):
            return 'Large'
        return 'Unknown'

    df_filtered['Location_Type'] = df_filtered['location_name'].apply(get_location_info)

    # Sort by population
    master_locations_order = [
        # Small
        'SaranacLake_NY_S',
        'FortBragg_CA_S',
        'Toccoa_GA_S',
        'Vidalia_GA_S',
        'Malone_NY_S',
        'Eureka_CA_S',

        # Mid-Size
        'Macon_GA_M',
        'Syracuse_NY_M',
        'Modesto_CA_M',

        # Large
        'Augusta_GA_L',
        'Buffalo_NY_L',
        'Atlanta_GA_L',
        'SanFrancisco_CA_L',
        'LA_CA_L',
        'NYC_NY_L'
    ]

    print("Sorted City Order:")
    print(master_locations_order)

    # Calculate Metrics
    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan])

        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25
        return pd.Series([p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['p10', 'p25', 'iqr']
    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low rating
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 1, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 1, 1, 0)

    categories = sorted(df_filtered['category'].unique())

    # Small, mid, large: blue, orange, green
    custom_colors = ['tab:blue', 'tab:orange', 'tab:green']

    # plot
    for category in categories:
        print(f"Category: {category}")
        cat_data = df_filtered[df_filtered['category'] == category]
        if cat_data.empty: continue

        current_order = [city for city in master_locations_order if city in cat_data['location_name'].unique()]

        # 1. Boxplots
        fig, ax = plt.subplots(figsize=(12, 8))
        hue_order = ['Small', 'Mid-Size', 'Large']

        sns.boxplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Average Rating by City (Sorted by Population)', fontsize=14)
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel("City (Small -> Large)")

        plt.tight_layout()
        plt.show()

        # 2. Violin
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.violinplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Avg Rating Distribution', fontsize=14)
        ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()


        # 3. Hist
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.histplot(data=subset, x='rating_value', bins=10, kde=False, stat='count',
                             color=custom_colors[i], ax=ax)
                ax.set_title(f'{loc_type} - Frequency')
                ax.set_xlabel('Rating')
                ax.set_ylabel('Count')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Rating Frequency', y=1.05)
        plt.tight_layout()
        plt.show()

        # 4. Normalized Hist
        print(f"[{category}] Histograms - Normalized")
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.histplot(data=subset, x='rating_value', bins=10, kde=False, stat='probability',
                             color=custom_colors[i], ax=ax)
                ax.set_title(f'{loc_type} - Normalized')
                ax.set_xlabel('Rating')
                ax.set_ylabel('Normalized Count')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Normalized Distribution', y=1.05)
        plt.tight_layout()
        plt.show()

        # 5. Scatter Plots
        print(f"[{category}] Scatter Plots (Avg vs P25)")
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

        for i, loc_type in enumerate(hue_order):
            ax = axes[i]
            subset = cat_data[cat_data['Location_Type'] == loc_type]

            if not subset.empty:
                sns.scatterplot(data=subset, x='rating_value', y='p25',
                                color=custom_colors[i], s=80, alpha=0.6, ax=ax)
                ax.set_title(f'{loc_type}')
                ax.set_xlabel('Average Rating')
                if i == 0:
                    ax.set_ylabel('25th Percentile')
            else:
                ax.set_title(f'{loc_type} (No Data)')

        plt.suptitle(f'[{category}] Scatter: Avg vs P25', y=1.05)
        plt.tight_layout()
        plt.show()

    # Low rating analysis
    print("Low Rating Analysis")

    hue_order_bar = ['Small', 'Mid-Size', 'Large']
    rename_dict = {0: 'High (>1)', 1: 'Low (<=1)'}

    # Avg <= 1
    low_avg_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (Avg Rating <= 1)")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Low Rating (Avg <= 1) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

    # P25 <= 1
    low_p25_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (25th Percentile <= 1)")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title('Low Rating (P25 <= 1) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("DataFrame is empty.")

# Regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from math import radians, cos, sin, asin, sqrt

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded data.")
except FileNotFoundError:
    print("File not found.")
    df = pd.DataFrame()

if not df.empty:
    # map zipcode
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Category (general, special, urgent/surgery)
    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
        print(df['Keyword_Category'].value_counts())
    else:
        print("Error: 'category' column missing")
        df = pd.DataFrame()

    df['Keyword_Category'] = pd.Categorical(
        df['Keyword_Category'],
        categories=['General', 'Special', 'Surgery'],
        ordered=False
    )

    # Location Size (Small/Mid/Large)
    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'],
        ordered=False
    )

    # Density (count per zipcode)
    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    # Distance to hub
    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1; dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 3956
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_miles'] = df.apply(get_distance, axis=1)

    # Preprocessing
    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))

    # Binary Outcome (Threshold <= 3.0)
    df['is_low_rating'] = np.where(df['rating_value'] <= 3.0, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=['rating_value', 'distance_miles', 'log_votes', 'zip_clinic_count', 'State', 'Keyword_Category', 'Location_Type'])
    print(f"Data ready. N = {len(reg_df)}")

    # Regression
    common_formula = """
    distance_miles + log_votes + zip_clinic_count +
    State + Keyword_Category + Location_Type
    """

    # OLS
    print("\nOLS (Rating Value)")
    model_ols = smf.ols(formula=f"rating_value ~ {common_formula}", data=reg_df).fit()
    print(model_ols.summary())

    # Logit
    print("\nLogit (Low Rating Risk <= 3.0)")
    logit_res = smf.logit(formula=f"is_low_rating ~ {common_formula}", data=reg_df).fit()
    print(logit_res.summary())

    # Average Marginal Effects (AME)
    print("\n Average Marginal Effects (AME)")
    mfx = logit_res.get_margeff(at='overall', method='dydx')
    print(mfx.summary())

    # Accuracy
    preds = logit_res.predict()
    prediction_binary = (preds > 0.5).astype(int)
    actual = reg_df['is_low_rating']
    accuracy = (prediction_binary == actual).mean()
    print(f"\nPercent Correctly Predicted: {accuracy:.2%}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from math import radians, cos, sin, asin, sqrt
import warnings

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded data.")
except FileNotFoundError:
    print("File not found.")
    df = pd.DataFrame()

if not df.empty:
    # Preprocessing

    # Map Zip to Location Name
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        # NY
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        # CA
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        # GA
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Category map
    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }
    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
    else:
        df['Keyword_Category'] = 'Unknown'

    # Location type
    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'], # Mid_Size is reference
        ordered=False
    )

    # 9-Group Category + Location
    df['Category_Location'] = df['Keyword_Category'].astype(str) + '_' + df['Location_Type'].astype(str)

    df['Category_Location'] = pd.Categorical(
        df['Category_Location'],
        categories=[
            'General_Mid_Size', # Reference
            'General_Large', 'General_Small',
            'Special_Mid_Size', 'Special_Large', 'Special_Small',
            'Surgery_Mid_Size', 'Surgery_Large', 'Surgery_Small'
        ],
        ordered=False
    )

    # Distance to hub
    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1; dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a)) # c= r * radian
            return c * 6371 # Earth's radius in km
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_to_hub'] = df.apply(get_distance, axis=1)

    # Avg dist to other clinics in same zip
    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    coords = df[['latitude', 'longitude', 'zip_clean']].dropna()

    def calc_peer_dist(sub):
        if len(sub) < 2: return 0.0
        # Convert to lat and lon to radians
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        # Build N*N matrix Dlatij = lati-latj; Dlonij = loni-lonj
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        # a = sin^2(dlat/2)+cos(lat1)*cos(lat2)*sin^2(dlon/2)
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dists = c * 6371
        # Sum rows, subtract self (0), divide by n-1
        return np.sum(dists, axis=1) / (len(sub) - 1)

    peer_dist_map = {}
    for z, group in coords.groupby('zip_clean'):
        dists = calc_peer_dist(group)
        # Store  dist
        for idx, val in zip(group.index, dists if isinstance(dists, np.ndarray) else [dists]*len(group)):
            peer_dist_map[idx] = val

    df['avg_peer_dist'] = df.index.map(peer_dist_map).fillna(0)

    # Preprocess
    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))
    df['is_low_rating'] = np.where(df['rating_value'] <= 3.0, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=[
        'rating_value', 'distance_to_hub', 'log_votes',
        'zip_clinic_count', 'State', 'Category_Location', 'avg_peer_dist', 'Location_Type'
    ]).copy()

    print(f"Data Ready. N = {len(reg_df)}")

    #  Regression

    # Model 1: 9-Group Dummies
    print("\nModel 1: 9-Group Type+Location")

    f1 = "rating_value ~ distance_to_hub + log_votes + State + Category_Location"
    f1_logit = f1.replace("rating_value", "is_low_rating")

    print("\nOLS Results")
    print(smf.ols(f1, data=reg_df).fit().summary())

    print("\nLogit AME")
    m1_logit = smf.logit(f1_logit, data=reg_df).fit(disp=0)
    print(m1_logit.get_margeff(at='overall', method='dydx').summary())


    # Model 2a: Avg distance without interact with zip count
    print("\nModel 2a: Avg distance without interact with zip count")

    f2a = """rating_value ~ distance_to_hub + log_votes + State +
             Category_Location +
             avg_peer_dist"""

    f2a_logit = f2a.replace("rating_value", "is_low_rating")

    print("\nOLS Results")
    print(smf.ols(f2a, data=reg_df).fit().summary())

    print("\nLogit AME")
    m2a_logit = smf.logit(f2a_logit, data=reg_df).fit(disp=0)
    print(m2a_logit.get_margeff(at='overall', method='dydx').summary())


    # Model 2b: Avg distance interact with zip count
    print("\nModel 2b: Avg distance interact with zip count")

    # Adding avg_peer_dist * zip_clinic_count
    f2b = """rating_value ~ distance_to_hub + log_votes + State +
             Category_Location +
             avg_peer_dist * zip_clinic_count"""

    f2b_logit = f2b.replace("rating_value", "is_low_rating")

    print("\nOLS Results")
    print(smf.ols(f2b, data=reg_df).fit().summary())

    print("\nLogit AME")
    m2b_logit = smf.logit(f2b_logit, data=reg_df).fit(disp=0)
    print(m2b_logit.get_margeff(at='overall', method='dydx').summary())


    # Model 3: City Size L/M/S interact with zip count
    print("\nModel 3: City Size L/M/S interact with zip count")

    # Adding Location_Type : zip_clinic_count interaction
    f3 = """rating_value ~ distance_to_hub + log_votes + State +
            Category_Location +
            avg_peer_dist  +  zip_clinic_count +
            Location_Type : zip_clinic_count"""

    f3_logit = f3.replace("rating_value", "is_low_rating")

    print("\nOLS Result")
    print(smf.ols(f3, data=reg_df).fit().summary())

    print("\nLogit AME")
    try:
        m3_logit = smf.logit(f3_logit, data=reg_df).fit(disp=0)
        print(m3_logit.get_margeff(at='overall', method='dydx').summary())
    except Exception as e:
        print(f"Logit AME Error: {e}")
        print("Tip: Check for perfect separation in 9-group dummies.")

else:
    print("DataFrame is empty.")

# Timeline

## Website

In [ ]:
!pip install beautifulsoup4 requests python-whois tldextract
!pip install duckduckgo-search

In [ ]:
import pandas as pd
import time
import random
from duckduckgo_search import DDGS
import whois
import tldextract
from datetime import datetime

def find_website_url(row):
    # query = f"{row['title']} {row['address']} dentist official site"
    query = f"{row['title']}"

    try:
        time.sleep(random.uniform(2, 4))
        with DDGS() as ddgs:

            results = list(ddgs.text(query, max_results=1))

        if results:
            url = results[0]

            blacklist = [
                'yelp.com', 'healthgrades.com', 'facebook.com',
                'mapquest.com', 'yellowpages.com', 'zocdoc.com',
                'linkedin.com', 'instagram.com', 'webmd.com'
            ]

            if any(b in url for b in blacklist):
                return None

            return url

    except Exception as e:
        print(f"Search error for {row['title']}: {e}")
        return None

    return None


def get_whois_creation_date(url):
    if not url: return None
    try:
        extracted = tldextract.extract(url)
        domain = f"{extracted.domain}.{extracted.suffix}"
        w = whois.whois(domain)
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            return creation_date[0]
        return creation_date
    except Exception as e:
        return None


input_file = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

try:
    df = pd.read_csv(input_file)
    print(f"Read {len(df)} records")
except FileNotFoundError:
    print("No file")
    exit()

# test
df = df.head(10).copy()


print("\nSearch for url")
urls = []
for index, row in df.iterrows():
    url = find_website_url(row)
    urls.append(url)
    if index % 10 == 0:
        print(f"  Processed {index + 1}... (Url: {url})")

df['website_url'] = urls

print("\nWHOIS regiter time...")
df['domain_creation_date'] = df['website_url'].apply(get_whois_creation_date)

df['domain_creation_date'] = pd.to_datetime(df['domain_creation_date'], utc=True, errors='coerce')
df['est_founding_year'] = df['domain_creation_date'].dt.year

print("\nResult:")
print(df[['title', 'website_url', 'est_founding_year']].head())

df.to_csv("final_processed_with_whois.csv", index=False)
print("\nFile Saved！")

## Reviews with timestamp

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import datetime
from tqdm import tqdm
import math

try:
    if not login or not password: raise NameError
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])
except NameError:
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])

LOCATION_MAP = {
    'Malone_NY_S': 1026588, 'SaranacLake_NY_S': 1023342, 'Syracuse_NY_M': 1027001,
    'Buffalo_NY_L': 1022764, 'NYC_NY_L': 1023191, 'Eureka_CA_S': 1013774,
    'FortBragg_CA_S': 1013806, 'Modesto_CA_M': 1014019, 'SanFrancisco_CA_L': 1014221,
    'LA_CA_L': 1013962, 'Vidalia_GA_S': 1015545, 'Toccoa_GA_S': 1015533,
    'Macon_GA_M': 1015427, 'Atlanta_GA_L': 1015254, 'Augusta_GA_L': 1015256
}

TEMP_DIR = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/reviews"
LOG_DIR = os.path.join(TEMP_DIR, "google_reviews_logs")
os.makedirs(LOG_DIR, exist_ok=True)

INPUT_ORIGINAL = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"
TEMP_SUMMARY_PATH = os.path.join(TEMP_DIR, "working_progress_summary.csv")

POST_URL = "https://api.dataforseo.com/v3/business_data/google/reviews/task_post"


def load_data_for_posting():
    if os.path.exists(TEMP_SUMMARY_PATH):
        print(f"Processing existing: {TEMP_SUMMARY_PATH}")
        df = pd.read_csv(TEMP_SUMMARY_PATH)
    elif os.path.exists(INPUT_ORIGINAL):
        print(f"Starting fresh from: {INPUT_ORIGINAL}")
        df = pd.read_csv(INPUT_ORIGINAL)
        df['scrape_status'] = 'pending'
        df['task_id'] = None
        df['earliest_review_time'] = None
        df['earliest_review_timestamp'] = None
        df['earliest_review_date'] = None
        df['total_reviews_scraped'] = 0
        df['used_location_code'] = None
        df['planned_depth'] = 0
        df.to_csv(TEMP_SUMMARY_PATH, index=False)
    else:
        print(f"Error: no file {INPUT_ORIGINAL}")
        return None, []

    if 'task_id' not in df.columns:
        df['task_id'] = None

    mask = (~df['scrape_status'].isin(['done', 'invalid_name', 'invalid_loc'])) & (df['task_id'].isna())
    todo_indices = df[mask].index.tolist()

    return df, todo_indices

def create_payload(df, indices):
    tasks = []
    valid_indices = []
    for idx in indices:
        row = df.loc[idx]
        title = str(row.get('title', '')).strip()
        loc_name = row.get('location_name')
        if pd.isna(loc_name): loc_name = row.get('search_location')
        loc_name = str(loc_name).strip()
        loc_code = LOCATION_MAP.get(loc_name)

        if not title or title.lower() == 'nan':
            df.at[idx, 'scrape_status'] = 'invalid_name'
            continue
        if not loc_code:
            df.at[idx, 'scrape_status'] = 'invalid_loc'
            continue


        try:
            votes = float(row.get('votes_count', 0))
            if pd.isna(votes): votes = 0
        except:
            votes = 0


        dynamic_depth = int(votes) + 50
        if dynamic_depth < 20: dynamic_depth = 20
        if dynamic_depth > 2000: dynamic_depth = 2000

        df.at[idx, 'used_location_code'] = loc_code
        df.at[idx, 'planned_depth'] = dynamic_depth

        tasks.append({
            "keyword": title,
            "location_code": loc_code,
            "language_code": "en",
            "depth": dynamic_depth,
            "sort_by": "newest",
            "tag": str(idx)
        })
        valid_indices.append(idx)
    return tasks, valid_indices

def save_json_log(data, prefix):
    ts = datetime.datetime.now().strftime("%H%M%S")
    path = os.path.join(LOG_DIR, f"{prefix}_{ts}.json")
    try:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    except: pass

def run_poster():
    main_df, todo_indices = load_data_for_posting()

    if not todo_indices:
        print("All tasks already posted or completed.")
        return

    BATCH_SIZE = 50
    batches = [todo_indices[i:i + BATCH_SIZE] for i in range(0, len(todo_indices), BATCH_SIZE)]
    print(f"Posting {len(todo_indices)} tasks in {len(batches)} batches...")

    for batch_indices in tqdm(batches, desc="Posting"):
        payload, valid_indices = create_payload(main_df, batch_indices)

        if not payload:
            main_df.to_csv(TEMP_SUMMARY_PATH, index=False)
            continue

        try:
            resp = requests.post(POST_URL, auth=CREDENTIALS, json=payload, timeout=60)
            data = resp.json()

            if data.get('status_code') == 20000:
                returned_tasks = data.get('tasks', [])
                if len(returned_tasks) == len(valid_indices):
                    for i, t in enumerate(returned_tasks):
                        if t.get('id'):
                            main_df.at[valid_indices[i], 'task_id'] = t['id']
                            main_df.at[valid_indices[i], 'scrape_status'] = 'posted'
                else:
                    print(f"Mismatch: sent {len(valid_indices)}, got {len(returned_tasks)}")

                save_json_log(data, "post_batch")
            else:
                print(f"API Error: {data.get('status_message')}")

        except Exception as e:
            print(f"Post Exception: {e}")

        main_df.to_csv(TEMP_SUMMARY_PATH, index=False)

    print("Posting finished. Task IDs saved.")


run_poster()

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import datetime
from tqdm import tqdm
import shutil


try:
    if not login or not password: raise NameError
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])
except NameError:
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])

TEMP_DIR = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/reviews"
TEMP_SUMMARY_PATH = os.path.join(TEMP_DIR, "working_progress_summary.csv")
TEMP_REVIEWS_PATH = os.path.join(TEMP_DIR, "working_reviews_part.csv")
FINAL_SUMMARY_OUTPUT = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed_with_earliest_review.csv"
FINAL_REVIEWS_OUTPUT = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv"

GET_URL = "https://api.dataforseo.com/v3/business_data/google/reviews/task_get"

def append_reviews_to_temp(data_list):
    if not data_list: return
    df_temp = pd.DataFrame(data_list)
    header = not os.path.exists(TEMP_REVIEWS_PATH)
    df_temp.to_csv(TEMP_REVIEWS_PATH, mode='a', header=header, index=False)

def run_getter():
    if not os.path.exists(TEMP_SUMMARY_PATH):
        print("Error: Summary file not found.")
        return

    df = pd.read_csv(TEMP_SUMMARY_PATH)

    cols_to_fix = ['earliest_review_time', 'earliest_review_timestamp', 'earliest_review_date']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = df[col].astype('object')

    mask = (df['task_id'].notna()) & (~df['scrape_status'].isin(['done', 'empty']))
    pending_df = df[mask]

    pending_ids = pending_df['task_id'].tolist()
    pending_map = dict(zip(pending_df['task_id'], pending_df.index))

    if not pending_ids:
        print("All tasks completed.")
    else:
        print(f"Fetching results for {len(pending_ids)} tasks...")

        chunk_size = 10
        chunks = [pending_ids[i:i + chunk_size] for i in range(0, len(pending_ids), chunk_size)]

        for chunk in tqdm(chunks, desc="Fetching"):
            batch_reviews_buffer = []
            updates_count = 0

            for tid in chunk:
                row_idx = pending_map[tid]

                try:
                    # Timeout 120s
                    get_resp = requests.get(f"{GET_URL}/{tid}", auth=CREDENTIALS, timeout=120)

                    if get_resp.status_code != 200:
                        continue

                    res_data = get_resp.json()

                    if res_data.get('status_code') == 20000:
                        status = 'failed'
                        count = 0

                        res_item = res_data['tasks'][0]['result']
                        if res_item and isinstance(res_item, list) and len(res_item) > 0:
                            items = res_item[0].get('items', []) or []
                            valid_items = [x for x in items if x.get('timestamp')]

                            if valid_items:
                                valid_items.sort(key=lambda x: x['timestamp'])

                                first = valid_items[0]
                                df.at[row_idx, 'earliest_review_time'] = first.get('time_ago')
                                df.at[row_idx, 'earliest_review_timestamp'] = first.get('timestamp')
                                df.at[row_idx, 'earliest_review_date'] = datetime.datetime.fromtimestamp(first.get('timestamp')).strftime('%Y-%m-%d')

                                status = 'done'
                                count = len(valid_items)

                                shop_info = df.loc[row_idx]
                                for r in valid_items:
                                    batch_reviews_buffer.append({
                                        'shop_idx': row_idx,
                                        'shop_title': shop_info.get('title'),
                                        'shop_zip': shop_info.get('zip'),
                                        'location_code': shop_info.get('used_location_code'),
                                        'rating': r.get('rating'),
                                        'timestamp': r.get('timestamp'),
                                        'date': datetime.datetime.fromtimestamp(r.get('timestamp')).strftime('%Y-%m-%d'),
                                        'review_text': r.get('review_text'),
                                        'profile_name': r.get('profile_name')
                                    })
                            else:
                                status = 'empty'
                        else:
                            status = 'empty'

                        df.at[row_idx, 'scrape_status'] = status
                        df.at[row_idx, 'total_reviews_scraped'] = count
                        updates_count += 1

                    elif res_data.get('status_code') == 40401:
                        # Task not ready
                        pass
                    else:
                        print(f"API Error {tid}: {res_data.get('status_message')}")

                except Exception as e:
                    pass

            if batch_reviews_buffer:
                append_reviews_to_temp(batch_reviews_buffer)

            if updates_count > 0:
                df.to_csv(TEMP_SUMMARY_PATH, index=False)

    print("\nSaving final files...")
    df.to_csv(FINAL_SUMMARY_OUTPUT, index=False)
    print(f"Summary Saved: {FINAL_SUMMARY_OUTPUT}")

    if os.path.exists(TEMP_REVIEWS_PATH):
        shutil.copy(TEMP_REVIEWS_PATH, FINAL_REVIEWS_OUTPUT)
        print(f"Reviews Saved: {FINAL_REVIEWS_OUTPUT}")
    else:
        print("No reviews collected.")


run_getter()

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import datetime
from tqdm import tqdm
import shutil
import glob

# --- Configuration ---
try:
    if not login or not password: raise NameError
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])
except NameError:
    CREDENTIALS = (os.environ["DATAFORSEO_LOGIN"], os.environ["DATAFORSEO_PASSWORD"])

TEMP_DIR = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/reviews"

GET_JSON_DIR = os.path.join(TEMP_DIR, "Get")
os.makedirs(GET_JSON_DIR, exist_ok=True)

TEMP_SUMMARY_PATH = os.path.join(TEMP_DIR, "working_progress_summary.csv")
TEMP_REVIEWS_PATH = os.path.join(TEMP_DIR, "working_reviews_part.csv")
FINAL_SUMMARY_OUTPUT = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed_with_earliest_review.csv"
FINAL_REVIEWS_OUTPUT = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv"

GET_URL = "https://api.dataforseo.com/v3/business_data/google/reviews/task_get"


def append_reviews_to_temp(data_list):
    if not data_list: return
    df_temp = pd.DataFrame(data_list)
    header = not os.path.exists(TEMP_REVIEWS_PATH)
    df_temp.to_csv(TEMP_REVIEWS_PATH, mode='a', header=header, index=False)

def parse_task_result(df, task_data, row_idx):
    """
    Parses the specific JSON structure provided by user.
    Input `task_data` is the 'result' list from the API response (or the whole JSON list).
    """
    batch_reviews = []


    items = []

    if isinstance(task_data, list) and len(task_data) > 0:
        if 'items' in task_data[0]:
            items = task_data[0]['items']

    elif isinstance(task_data, dict) and 'result' in task_data:
         if task_data['result'] and isinstance(task_data['result'], list):
             if 'items' in task_data['result'][0]:
                 items = task_data['result'][0]['items']

    elif isinstance(task_data, dict) and 'tasks' in task_data:
        if task_data['tasks'] and task_data['tasks'][0].get('result'):
             items = task_data['tasks'][0]['result'][0].get('items', [])

    if not items:
        df.at[row_idx, 'scrape_status'] = 'empty'
        df.at[row_idx, 'total_reviews_scraped'] = 0
        return [], True

    valid_items = [x for x in items if x.get('timestamp')]

    if valid_items:
        valid_items.sort(key=lambda x: x['timestamp'])

        first = valid_items[0]
        df.at[row_idx, 'earliest_review_time'] = first.get('time_ago')
        df.at[row_idx, 'earliest_review_timestamp'] = first.get('timestamp')
        df.at[row_idx, 'earliest_review_date'] = datetime.datetime.fromtimestamp(first.get('timestamp')).strftime('%Y-%m-%d')

        df.at[row_idx, 'scrape_status'] = 'done'
        df.at[row_idx, 'total_reviews_scraped'] = len(valid_items)

        shop_info = df.loc[row_idx]

        for r in valid_items:
            rating_val = None
            if r.get('rating') and isinstance(r['rating'], dict):
                rating_val = r['rating'].get('value')
            else:
                rating_val = r.get('rating')

            batch_reviews.append({
                'shop_idx': row_idx,
                'shop_title': shop_info.get('title'),
                'shop_zip': shop_info.get('zip'),
                'location_code': shop_info.get('used_location_code'),
                'rating': rating_val,
                'timestamp': r.get('timestamp'),
                'date': datetime.datetime.fromtimestamp(r.get('timestamp')).strftime('%Y-%m-%d'),
                'review_text': r.get('review_text'), # Directly use review_text as per JSON
                'profile_name': r.get('profile_name'),
                'review_url': r.get('review_url')
            })
        return batch_reviews, True
    else:
        df.at[row_idx, 'scrape_status'] = 'empty'
        df.at[row_idx, 'total_reviews_scraped'] = 0
        return [], True

def run_getter():
    if not os.path.exists(TEMP_SUMMARY_PATH):
        print("Error: Summary file not found.")
        return

    df = pd.read_csv(TEMP_SUMMARY_PATH)

    # Fix types
    cols_to_fix = ['earliest_review_time', 'earliest_review_timestamp', 'earliest_review_date']
    for col in cols_to_fix:
        if col in df.columns: df[col] = df[col].astype('object')

    valid_tasks = df[df['task_id'].notna()]
    task_id_to_idx = dict(zip(valid_tasks['task_id'], valid_tasks.index))

    json_files = glob.glob(os.path.join(GET_JSON_DIR, "*.json"))
    print(f"Scanning {len(json_files)} local files in {GET_JSON_DIR}...")

    local_recovered = 0

    for jf in tqdm(json_files, desc="Local Scan"):
        try:
            # Extract TID from filename task_get_{tid}.json or just {tid}.json
            fname = os.path.basename(jf)
            tid_from_name = fname.replace("task_get_", "").replace(".json", "")

            if tid_from_name in task_id_to_idx:
                row_idx = task_id_to_idx[tid_from_name]
                current_status = df.at[row_idx, 'scrape_status']

                if current_status not in ['done', 'empty']:
                    with open(jf, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    reviews, success = parse_task_result(df, data, row_idx)
                    if success:
                        append_reviews_to_temp(reviews)
                        local_recovered += 1
        except Exception:
            pass

    if local_recovered > 0:
        print(f"Recovered {local_recovered} tasks from local.")
        df.to_csv(TEMP_SUMMARY_PATH, index=False)

    mask = (df['task_id'].notna()) & (~df['scrape_status'].isin(['done', 'empty']))
    pending_df = df[mask]
    pending_ids = pending_df['task_id'].tolist()

    if not pending_ids:
        print("All tasks completed!")
    else:
        print(f"Fetching {len(pending_ids)} tasks from API...")

        chunk_size = 10
        chunks = [pending_ids[i:i + chunk_size] for i in range(0, len(pending_ids), chunk_size)]

        for chunk in tqdm(chunks, desc="API Fetch"):
            batch_buffer = []
            updates = 0

            for tid in chunk:
                row_idx = task_id_to_idx[tid]
                try:
                    resp = requests.get(f"{GET_URL}/{tid}", auth=CREDENTIALS, timeout=120)
                    if resp.status_code != 200: continue

                    res_data = resp.json()

                    if res_data.get('status_code') == 20000:
                        task_obj = res_data['tasks'][0]

                        reviews, success = parse_task_result(df, task_obj, row_idx)

                        if success:
                            batch_buffer.extend(reviews)
                            updates += 1

                            save_path = os.path.join(GET_JSON_DIR, f"{tid}.json")
                            with open(save_path, 'w', encoding='utf-8') as f:
                                json.dump(res_data, f, ensure_ascii=False)

                    elif res_data.get('status_code') == 40401:
                        pass
                except Exception:
                    pass

            if batch_buffer:
                append_reviews_to_temp(batch_buffer)
            if updates > 0:
                df.to_csv(TEMP_SUMMARY_PATH, index=False)

    print("\nSaving final files...")
    df.to_csv(FINAL_SUMMARY_OUTPUT, index=False)
    if os.path.exists(TEMP_REVIEWS_PATH):
        shutil.copy(TEMP_REVIEWS_PATH, FINAL_REVIEWS_OUTPUT)

run_getter()

## Merge

In [ ]:
import pandas as pd
import numpy as np


df_timeline = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/timeline_data_final.csv')
df_processed = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed_with_earliest_review.csv')

temp_year_str = df_timeline['est_founded'].astype(str).str.replace(r'\.0$', '', regex=True)

# est_founded change to datetime Year-12-31
df_timeline['temp_founded_dt'] = pd.to_datetime(temp_year_str + '-12-31', errors='coerce')

# left join
df_merged = pd.merge(
    df_processed,
    df_timeline[['title', 'est_founded', 'temp_founded_dt']],
    on='title',
    how='left'
)

df_merged['temp_review_dt'] = pd.to_datetime(df_merged['earliest_review_date'], errors='coerce')

# earliest
df_merged['start_date'] = df_merged[['temp_founded_dt', 'temp_review_dt']].min(axis=1)

df_final = df_merged.drop(columns=['temp_founded_dt', 'temp_review_dt'])

df_final.to_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv', index=False)

print(df_final[['title', 'est_founded', 'earliest_review_date', 'start_date']].head())

## Count

In [ ]:
total_records = len(df_final)

# not nan count
count_founded = df_final['est_founded'].notna().sum()
count_review = df_final['earliest_review_date'].notna().sum()
count_start = df_final['start_date'].notna().sum()

# not nan percentage
pct_founded = (count_founded / total_records) * 100
pct_review = (count_review / total_records) * 100
pct_start = (count_start / total_records) * 100


print(f"Total Records: {total_records}")
print(f"1. est_founded (Website Founded Time):        {count_founded} ({pct_founded:.2f}%)")
print(f"2. earliest_review_date (First Review Time): {count_review} ({pct_review:.2f}%)")
print(f"3. start_date (Combined):     {count_start} ({pct_start:.2f}%)")

improvement = count_start - count_review
print(f"\nImprovement: By merging, gained {improvement} valid start dates.")

In [ ]:
rating_col = 'rating' if 'rating' in df_final.columns else 'rating_value'

bins = [0.9, 1.9, 2.9, 3.9, 5.1]
labels = ['1-2', '2-3', '3-4', '4-5']

df_final['rating_range'] = pd.cut(df_final[rating_col], bins=bins, labels=labels, include_lowest=True)

def analyze_time_coverage(group):
    return pd.Series({
        'Total_Count': len(group),

        'Web_have': group['est_founded'].notna().sum(),
        'Web_NaN': group['est_founded'].isna().sum(),

        'Review_gave': group['earliest_review_date'].notna().sum(),
        'Review_NaN': group['earliest_review_date'].isna().sum(),

        'Merge_have': group['start_date'].notna().sum(),
        'Merge_NaN': group['start_date'].isna().sum()
    })

range_stats = df_final.groupby('rating_range', observed=False).apply(analyze_time_coverage)

range_stats['Merge_Coverage_%'] = (range_stats['Merge_have'] / range_stats['Total_Count'] * 100).round(2)

print("--- Each rate captured time ---")
print(range_stats)

# Analysis

## 1 star

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

df = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv')

if not df.empty:
    # Map Zip to Location Name
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    # Use 'zip' as per your CSV columns
    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        # NY
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        # CA
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        # GA
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Category map
    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
    else:
        df['Keyword_Category'] = 'Unknown'

    # Location type
    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'], # Mid_Size is reference
        ordered=False
    )

    # 9-Group Category + Location
    df['Category_Location'] = df['Keyword_Category'].astype(str) + '_' + df['Location_Type'].astype(str)

    df['Category_Location'] = pd.Categorical(
        df['Category_Location'],
        categories=[
            'General_Mid_Size', # Reference
            'General_Large', 'General_Small',
            'Special_Mid_Size', 'Special_Large', 'Special_Small',
            'Surgery_Mid_Size', 'Surgery_Large', 'Surgery_Small'
        ],
        ordered=False
    )

    # Distance to hub
    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 6371 # Earth's radius in km
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_to_hub'] = df.apply(get_distance, axis=1)

    # Peer Distance
    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    def calc_peer_dist(sub):
        if len(sub) < 2: return 0.0
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dists = c * 6371
        # Sum distances, divide by N-1 to get average distance to others
        return np.sum(dists, axis=1) / (len(sub) - 1)

    coords = df[['latitude', 'longitude', 'zip_clean']].dropna()
    peer_dist_map = {}
    for z, group in coords.groupby('zip_clean'):
        if len(group) > 1:
            dists = calc_peer_dist(group)
            for idx, val in zip(group.index, dists):
                peer_dist_map[idx] = val
        else:
            for idx in group.index:
                peer_dist_map[idx] = 0.0

    df['avg_peer_dist'] = df.index.map(peer_dist_map).fillna(0)

    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
    df_1star = df[df['rating_value'] <= 1.5].copy()

    peer_dist_1star_map = {}

    if not df_1star.empty:
        coords_1star = df_1star[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_1star.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_1star_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_1star_map[idx] = 0.0 # Only 1 bad clinic in this zip, no peers

    df['avg_dist_to_1star_peers'] = df.index.map(peer_dist_1star_map)


    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))
    df['is_low_rating'] = np.where(df['rating_value'] < 2, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=[
        'rating_value', 'distance_to_hub', 'log_votes',
        'zip_clinic_count', 'State', 'Category_Location', 'avg_peer_dist', 'Location_Type'
    ]).copy()

    print(f"N = {len(reg_df)}")

    subset_1star = df[df['rating_value'] < 2]


    # Distance to Hub
    mean_hub_1star = subset_1star['distance_to_hub'].mean()
    mean_hub_all = df['distance_to_hub'].mean()
    print(f"Avg Dist to Hub:")
    print(f"   - 1-Star Clinics: {mean_hub_1star:.2f} km")
    print(f"   - All Clinics:    {mean_hub_all:.2f} km")

    # Distance to peers
    mean_peer_1star = subset_1star['avg_dist_to_1star_peers'].mean()
    # For all: Average distance to ALL other clinics
    mean_peer_all = df['avg_peer_dist'].mean()

    print(f"Avg Dist to Peers:")
    print(f"   - 1-Star Clinics (dist to other 1-stars): {mean_peer_1star:.2f} km")
    print(f"   - All Clinics (dist to all peers):        {mean_peer_all:.2f} km")


## Skewed Distribution Testing

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Scaled distribution test
vars_to_check = ['distance_to_hub', 'votes_count', 'avg_peer_dist', 'zip_clinic_count']

regression_features = []
final_col_map = {}
for col in vars_to_check:
    if col not in df.columns:
        print(f"Skipping {col} (Not found)")
        continue

    series = pd.to_numeric(df[col], errors='coerce').dropna()
    if len(series) == 0: continue

    mean_v = series.mean()
    median_v = series.median()
    max_v = series.max()

    print(f"Variable: {col}")
    print(f"  Mean: {mean_v:.2f}, Median: {median_v:.2f}, Max: {max_v:.2f}")

    # mean >> median
    if (mean_v > 2 * median_v) or (max_v > 10 * mean_v):
        print(f"Skewed distribution. Applying log transformation.")

        # log
        log_col_name = f'log_{col}'
        df[log_col_name] = np.log1p(df[col].fillna(0))

        # new feature
        regression_features.append(log_col_name)
        final_col_map[col] = log_col_name
    else:
        print(f"Not skewed distribution. Keeping original.")
        regression_features.append(col)
        final_col_map[col] = col

categorical_features = ['Location_Type', 'Keyword_Category']
for cat in categorical_features:
    if cat in df.columns:
        regression_features.append(f"C({cat})")



## Quantile regression

In [ ]:
print(f"Features for Regression: {regression_features}")

rating_col = 'rating_value' if 'rating_value' in df.columns else 'rating'
df[rating_col] = pd.to_numeric(df[rating_col], errors='coerce')

valid_cols = [c for c in regression_features if not c.startswith('C(')]
reg_data = df.dropna(subset=valid_cols + [rating_col]).copy()

f_dist = final_col_map.get('distance_to_hub', 'distance_to_hub')
f_votes = final_col_map.get('votes_count', 'votes_count')
f_peer = final_col_map.get('avg_peer_dist', 'avg_peer_dist')
f_zip_count = final_col_map.get('zip_clinic_count', 'zip_clinic_count')

# Drop NA
valid_cols = [c for c in regression_features if not c.startswith('C(')]
reg_data = df.dropna(subset=valid_cols + [rating_col]).copy()


def print_summary(res, model_name, q):
    print(f"Model: {model_name} (Quantile: {q})")

    try:
        prsquared = res.prsquared
    except:
        prsquared = np.nan

    n = res.nobs
    k = res.df_model
    df_resid = res.df_resid

    if df_resid > 0:
        adj_prsquared = 1 - (1 - prsquared) * (n - 1) / df_resid
    else:
        adj_prsquared = np.nan

    print(f"Adj. R-squared:   {adj_prsquared:.4f}")
    print(f"No. Observations:   {int(res.nobs)}")
    print(f"Df Residuals:       {int(res.df_resid)}")
    print(f"Df Model:           {int(res.df_model)}")

    print(res.summary().tables[1])

def run_quantreg(name, formula, data):
    print(f"\n{name}")
    print(f"Formula: {formula}")
    try:
        # q=0.5
        mod_med = smf.quantreg(formula, data)
        res_med = mod_med.fit(q=0.5, max_iter=2000)
        print_summary(res_med, name, 0.5)

        # q=0.1
        mod_low = smf.quantreg(formula, data)
        res_low = mod_low.fit(q=0.1, max_iter=2000)
        print_summary(res_low, name, 0.1)

    except Exception as e:
        print(f"Error in {name}: {e}")

# --- Model 1: 9-Group Type+Location ---
f1 = f"{rating_col} ~ {f_dist} + {f_votes} + State + Category_Location"
run_quantreg("Model 1: 9-Group Type+Location", f1, reg_data)

# --- Model 2a: Avg Peer Dist---
f2a = f"""{rating_col} ~ {f_dist} + {f_votes} + State +
          Category_Location + {f_peer}"""
run_quantreg("Model 2a: Avg Peer Dist", f2a, reg_data)

# --- Model 2b: Peer Dist * Zip Count Interaction ---
f2b = f"""{rating_col} ~ {f_dist} + {f_votes} + State +
          Category_Location + {f_peer} * {f_zip_count}"""
run_quantreg("Model 2b: Peer Dist * Zip Count Interaction", f2b, reg_data)

# --- Model 3: City Size * Zip Count Interaction ---
f3 = f"""{rating_col} ~ {f_dist} + {f_votes} + State +
         Category_Location + {f_peer} + {f_zip_count} +
         Location_Type : {f_zip_count}"""
run_quantreg("Model 3: City Size * Zip Count Interaction", f3, reg_data)

## Time

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from math import radians, sin, cos, sqrt, asin
import warnings
warnings.filterwarnings("ignore")

print("Loading Data with earliest date...")
try:
    # original data with only last date
    df_static = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv')
    print(f"Loaded {len(df_static)} rows from static file.")

    # title & zip clean
    if 'title' in df_static.columns:
        df_static['title'] = df_static['title'].astype(str).str.strip()

    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df_static.columns:
        df_static['zip_clean'] = df_static['zip'].apply(clean_zip_code)
    else:
        df_static['zip_clean'] = np.nan

    # Deduplicate (Keep first)

    if 'start_date' in df_static.columns:
        df_static['start_date'] = pd.to_datetime(df_static['start_date'], errors='coerce')
        df_static = df_static.sort_values('start_date')

    df_static = df_static.drop_duplicates(subset=['title', 'zip_clean'], keep='first')

    title_counts = df_static['title'].value_counts()
    dupe_titles = set(title_counts[title_counts > 1].index)
    print(f"Static Data after deduplication: {len(df_static)} rows.")
    print(f"Unique Titles: {len(df_static) - len(dupe_titles)}")
    print(f"Duplicate Titles: {len(dupe_titles)}")

    # distance to hub
    print("distance_to_hub")
    print(f"Rows before distance calc: {len(df_static)}") # NEW: Show count before

    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except: return 'Unknown'
        if 10000 <= z <= 14999: return 'NYC_NY_L'
        if 90000 <= z <= 96000: return 'LA_CA_L'
        if 30000 <= z <= 39999: return 'Atlanta_GA_L'
        return 'Unknown'

    if 'mapped_location' not in df_static.columns:
        df_static['mapped_location'] = df_static['zip_clean'].apply(map_zip_to_location)

    def get_distance(row):
        if 'distance_to_hub' in row and pd.notna(row['distance_to_hub']): return row['distance_to_hub']
        loc = row.get('mapped_location', 'Unknown')
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1; dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 6371
        except: return np.nan

    if 'latitude' in df_static.columns:
        df_static['distance_to_hub'] = df_static.apply(get_distance, axis=1)
    else:
        df_static['distance_to_hub'] = np.nan

    print(f"Rows with valid distance: {df_static['distance_to_hub'].count()}") # NEW: Show count after

    # Categories
    cat_mapping = {'keywords_General_Dentist': 'General', 'keywords_Special_Dentist': 'Special', 'keywords_Surgery_Dentist': 'Surgery'}
    cat_col = next((c for c in df_static.columns if 'category' in c.lower()), None)
    if cat_col:
        df_static['Keyword_Category'] = df_static[cat_col].map(cat_mapping).fillna('Other')
    else:
        df_static['Keyword_Category'] = 'Unknown'

    # Location & State
    def get_loc_type(loc):
        s = str(loc)
        if s.endswith('_S'): return 'Small'
        if s.endswith('_M'): return 'Mid_Size'
        if s.endswith('_L'): return 'Large'
        return 'Unknown'
    df_static['Location_Type'] = df_static['mapped_location'].apply(get_loc_type)

    def get_state(loc):
        s = str(loc)
        if '_NY_' in s: return 'NY'
        if '_CA_' in s: return 'CA'
        if '_GA_' in s: return 'GA'
        return 'Other'
    df_static['State'] = df_static['mapped_location'].apply(get_state)

    df_static['Category_Location'] = df_static['Keyword_Category'].astype(str) + '_' + df_static['Location_Type'].astype(str)

    # Peer distance (Density)
    def calc_peer_dist(sub):
        if len(sub) < 2: return 0.0
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return np.sum(c * 6371, axis=1) / (len(sub) - 1)

    peer_dist_map = {}
    if 'latitude' in df_static.columns:
        coords = df_static[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_map[idx] = 0.0
    df_static['avg_peer_dist'] = df_static.index.map(peer_dist_map).fillna(0)

    # Log votes
    votes_col = next((c for c in df_static.columns if 'votes' in c.lower()), 'votes_count')
    if votes_col in df_static.columns:
        df_static[votes_col] = pd.to_numeric(df_static[votes_col], errors='coerce').fillna(0)
        df_static['log_votes'] = np.log1p(df_static[votes_col])
    else:
        df_static['log_votes'] = 0

except FileNotFoundError:
    print("Error: final_merged_time.csv not found.")
    df_static = pd.DataFrame()

if not df_static.empty:

    review_file = '/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv'
    chunk_size = 50000
    shop_year_stats = []

    cols_to_keep = ['start_date', 'distance_to_hub', 'zip_clean',
                    'avg_peer_dist', 'log_votes', 'State', 'Category_Location']

    for col in cols_to_keep:
        if col not in df_static.columns: df_static[col] = np.nan

    # Lookup by title for unique clinic titles
    df_unique = df_static[~df_static['title'].isin(dupe_titles)]
    lookup_unique = df_unique.set_index('title')[cols_to_keep].to_dict('index')

    # Lookup by title & zip for duplicated titles
    df_chain = df_static[df_static['title'].isin(dupe_titles)]
    lookup_chain = df_chain.set_index(['title', 'zip_clean'], drop=False)[cols_to_keep].to_dict('index')

    try:
        chunk_iter = pd.read_csv(review_file, chunksize=chunk_size)
        total_rows = 0

        for i, chunk in enumerate(chunk_iter):
            total_rows += len(chunk)
            if i % 10 == 0: print(f"Processing chunk {i} (Total: {total_rows})...")

            if 'shop_title' not in chunk.columns or 'shop_zip' not in chunk.columns: continue

            chunk['title_clean'] = chunk['shop_title'].astype(str).str.strip()
            chunk['zip_clean'] = chunk['shop_zip'].astype(str).str.replace(r'[^\d]', '', regex=True).str[:5]
            chunk['zip_clean'] = pd.to_numeric(chunk['zip_clean'], errors='coerce')

            def match_shop(row):
                t = row['title_clean']
                z = row['zip_clean']

                if t in dupe_titles:
                    meta = lookup_chain.get((t, z))
                    if meta: meta['match_type'] = 'chain_strict'
                    return meta

                else:
                    meta = lookup_unique.get(t)
                    if meta:
                        meta['match_type'] = 'unique_lax'
                    return meta

            chunk['meta'] = chunk.apply(match_shop, axis=1)
            chunk = chunk.dropna(subset=['meta'])

            if chunk.empty: continue

            meta_df = pd.DataFrame(chunk['meta'].tolist(), index=chunk.index)

            chunk['official_start'] = meta_df['start_date']
            chunk['distance_to_hub'] = meta_df['distance_to_hub']
            chunk['static_zip'] = meta_df['zip_clean']
            chunk['avg_peer_dist'] = meta_df['avg_peer_dist']
            chunk['log_votes'] = meta_df['log_votes']
            chunk['State'] = meta_df['State']
            chunk['Category_Location'] = meta_df['Category_Location']

            # Date
            date_col = 'date' if 'date' in chunk.columns else 'timestamp'
            chunk[date_col] = pd.to_datetime(chunk[date_col], errors='coerce')
            chunk['review_year'] = chunk[date_col].dt.year
            chunk['rating'] = pd.to_numeric(chunk['rating'], errors='coerce')
            # ID: title + zip
            chunk['shop_id'] = chunk['title_clean'] + "_" + chunk['static_zip'].astype(str)

            agg = chunk.groupby(['shop_id', 'review_year']).agg({
                'rating': ['sum', 'count'],
                'static_zip': 'first',
                'distance_to_hub': 'first',
                'official_start': 'first',
                'avg_peer_dist': 'first',
                'log_votes': 'first',
                'State': 'first',
                'Category_Location': 'first'
            })
            agg.columns = ['sum_rating', 'count_reviews', 'zip_clean', 'distance_to_hub', 'official_start',
                           'avg_peer_dist', 'log_votes', 'State', 'Category_Location']
            agg = agg.reset_index()
            shop_year_stats.append(agg)

        if not shop_year_stats:
            print("No matches found.")
        else:
            full_panel = pd.concat(shop_year_stats, ignore_index=True)
            print(f"Total Shop-Years: {len(full_panel)}")

            # Aggregation
            final_panel = full_panel.groupby(['shop_id', 'review_year']).agg({
                'sum_rating': 'sum',
                'count_reviews': 'sum',
                'zip_clean': 'first',
                'distance_to_hub': 'first',
                'official_start': 'first',
                'avg_peer_dist': 'first',
                'log_votes': 'first',
                'State': 'first',
                'Category_Location': 'first'
            }).reset_index()

            final_panel['yearly_rating'] = final_panel['sum_rating'] / final_panel['count_reviews']

            # Fix Dates
            min_years = final_panel.groupby('shop_id')['review_year'].min().reset_index(name='proxy_start')
            final_panel = pd.merge(final_panel, min_years, on='shop_id', how='left')

            final_panel['official_start'] = pd.to_datetime(final_panel['official_start'], errors='coerce')
            final_panel['final_start'] = final_panel['official_start'].dt.year.fillna(final_panel['proxy_start'])

            final_panel['clinic_age'] = final_panel['review_year'] - final_panel['final_start']
            final_panel = final_panel[final_panel['clinic_age'] >= 0]

            print(f"Total Valid Panel Rows (Age >= 0): {len(final_panel)}")

            # Lagged Density
            density = final_panel.groupby(['zip_clean', 'review_year'])['shop_id'].nunique().reset_index(name='active_clinics')
            final_panel = pd.merge(final_panel, density, on=['zip_clean', 'review_year'], how='left')

            density['next'] = density['review_year'] + 1
            final_panel = pd.merge(final_panel, density[['zip_clean', 'next', 'active_clinics']],
                                   left_on=['zip_clean', 'review_year'], right_on=['zip_clean', 'next'], how='left')
            final_panel.rename(columns={'active_clinics_y': 'lagged_density'}, inplace=True)
            final_panel['lagged_density'] = final_panel['lagged_density'].fillna(final_panel['active_clinics_x'])

            # Model A: Panel
            print("\nModel A: Panel")
            # Added new features to subset and formula
            reg_cols = ['yearly_rating', 'clinic_age', 'lagged_density', 'distance_to_hub',
                        'avg_peer_dist', 'log_votes', 'State', 'Category_Location']
            reg_data = final_panel.dropna(subset=reg_cols)
            print(f"Data available for regression: {len(reg_data)}")

            if len(reg_data) > 100:
                mod = smf.ols("yearly_rating ~ clinic_age + I(clinic_age**2) + lagged_density + distance_to_hub + avg_peer_dist + log_votes + C(State) + C(Category_Location)",
                              data=reg_data).fit(cov_type='cluster', cov_kwds={'groups': reg_data['zip_clean']})
                print(mod.summary().tables[1])
                print(f"Adj. R-squared: {mod.rsquared_adj:.4f}")

                # Check Shape
                age_sq = mod.params.get('I(clinic_age ** 2)', 0)
                print(f"Age^2 Coef: {age_sq:.5f} ({'Inverted-U' if age_sq < 0 else 'U-Shape'})")
            else:
                print("Not enough data for Panel Regression.")

            # Model B: Survival
            print("\nModel B: Survival Analysis (Rating < 4)")
            THRESHOLD = 4

            final_panel.sort_values(['shop_id', 'review_year'], inplace=True)
            failures = final_panel[final_panel['yearly_rating'] < THRESHOLD].groupby('shop_id')['review_year'].min().reset_index(name='fail_year')

            surv_df = final_panel.groupby('shop_id').agg({
                'final_start': 'min', 'distance_to_hub': 'first', 'review_year': 'max',
                'avg_peer_dist': 'first', 'log_votes': 'first' # Add features for Survival too? Formula uses them below
            }).reset_index()

            surv_df = pd.merge(surv_df, failures, on='shop_id', how='left')
            surv_df['event'] = np.where(surv_df['fail_year'].notna(), 1, 0)

            surv_df['end_time'] = np.where(surv_df['event']==1, surv_df['fail_year'], surv_df['review_year'])
            surv_df['time'] = surv_df['end_time'] - surv_df['final_start']

            surv_data = surv_df[(surv_df['time'] > 0) & (surv_df['distance_to_hub'].notna())]
            print(f"Survival Events: {surv_data['event'].sum()} / {len(surv_data)}")

            if surv_data['event'].sum() > 10:
                cph = sm.PHReg.from_formula("time ~ distance_to_hub + avg_peer_dist + log_votes", status=surv_data['event'], data=surv_data)
                res = cph.fit()
                print(res.summary().tables[1])

                hr = np.exp(res.params['distance_to_hub'])
            else:
                print("Not enough events.")

    except Exception as e:
        print(f"Error: {e}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from math import radians, sin, cos, sqrt, asin
import warnings
warnings.filterwarnings("ignore")

# Quantile
try:
    df_static = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv')
    print(f"Loaded {len(df_static)} rows.")

    if 'title' in df_static.columns:
        df_static['title'] = df_static['title'].astype(str).str.strip()

    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    zip_col = next((col for col in df_static.columns if 'zip' in col.lower()), None)
    if zip_col:
        df_static['zip_clean'] = df_static[zip_col].apply(clean_zip_code)
    else:
        df_static['zip_clean'] = np.nan

    df_static = df_static.drop_duplicates(subset=['title', 'zip_clean'], keep='first')

    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except: return 'Unknown'
        if 10000 <= z <= 14999: return 'NYC_NY_L'
        if 90000 <= z <= 96000: return 'LA_CA_L'
        if 30000 <= z <= 39999: return 'Atlanta_GA_L'
        return 'Unknown'

    if 'mapped_location' not in df_static.columns:
        df_static['mapped_location'] = df_static['zip_clean'].apply(map_zip_to_location)

    def get_distance(row):
        if 'distance_to_hub' in row and pd.notna(row['distance_to_hub']): return row['distance_to_hub']
        loc = row.get('mapped_location', 'Unknown')
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1; dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 6371
        except: return np.nan

    if 'latitude' in df_static.columns:
        df_static['distance_to_hub'] = df_static.apply(get_distance, axis=1)
    else:
        df_static['distance_to_hub'] = np.nan

    cat_mapping = {'keywords_General_Dentist': 'General', 'keywords_Special_Dentist': 'Special', 'keywords_Surgery_Dentist': 'Surgery'}
    cat_col = next((c for c in df_static.columns if 'category' in c.lower()), None)
    if cat_col:
        df_static['Keyword_Category'] = df_static[cat_col].map(cat_mapping).fillna('Other')
    else:
        df_static['Keyword_Category'] = 'Unknown'

    def get_loc_type(loc):
        s = str(loc)
        if s.endswith('_S'): return 'Small'
        if s.endswith('_M'): return 'Mid_Size'
        if s.endswith('_L'): return 'Large'
        return 'Unknown'
    df_static['Location_Type'] = df_static['mapped_location'].apply(get_loc_type)

    def get_state(loc):
        s = str(loc)
        if '_NY_' in s: return 'NY'
        if '_CA_' in s: return 'CA'
        if '_GA_' in s: return 'GA'
        return 'Other'
    df_static['State'] = df_static['mapped_location'].apply(get_state)

    df_static['Category_Location'] = df_static['Keyword_Category'].astype(str) + '_' + df_static['Location_Type'].astype(str)

    def calc_peer_dist(sub):
        if len(sub) < 2: return 0.0
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return np.sum(c * 6371, axis=1) / (len(sub) - 1)

    peer_dist_map = {}
    if 'latitude' in df_static.columns:
        coords = df_static[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_map[idx] = 0.0
    df_static['avg_peer_dist'] = df_static.index.map(peer_dist_map).fillna(0)

    votes_col = next((c for c in df_static.columns if 'votes' in c.lower()), 'votes_count')
    if votes_col in df_static.columns:
        df_static[votes_col] = pd.to_numeric(df_static[votes_col], errors='coerce').fillna(0)
        df_static['log_votes'] = np.log1p(df_static[votes_col])
    else:
        df_static['log_votes'] = 0

except FileNotFoundError:
    print("Error: final_merged_time.csv not found.")
    df_static = pd.DataFrame()

if not df_static.empty:

    review_file = '/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv'
    chunk_size = 50000
    shop_year_stats = []

    title_counts = df_static['title'].value_counts()
    dupe_titles = set(title_counts[title_counts > 1].index)

    cols_to_keep = ['start_date', 'distance_to_hub', 'zip_clean', 'avg_peer_dist',
                    'log_votes', 'State', 'Category_Location', 'Location_Type']

    for col in cols_to_keep:
        if col not in df_static.columns: df_static[col] = np.nan

    df_unique = df_static[~df_static['title'].isin(dupe_titles)]
    lookup_unique = df_unique.set_index('title')[cols_to_keep].to_dict('index')

    df_chain = df_static[df_static['title'].isin(dupe_titles)]
    lookup_chain = df_chain.set_index(['title', 'zip_clean'], drop=False)[cols_to_keep].to_dict('index')

    try:
        chunk_iter = pd.read_csv(review_file, chunksize=chunk_size)
        total_rows = 0

        for i, chunk in enumerate(chunk_iter):
            total_rows += len(chunk)
            if i % 10 == 0: print(f"Processing chunk {i} (Total reviews: {total_rows})...")

            if 'shop_title' not in chunk.columns or 'shop_zip' not in chunk.columns: continue

            chunk['title_clean'] = chunk['shop_title'].astype(str).str.strip()
            chunk['zip_clean'] = chunk['shop_zip'].astype(str).str.replace(r'[^\d]', '', regex=True).str[:5]
            chunk['zip_clean'] = pd.to_numeric(chunk['zip_clean'], errors='coerce')

            def match_shop(row):
                t = row['title_clean']
                z = row['zip_clean']
                if t in dupe_titles:
                    return lookup_chain.get((t, z))
                else:
                    return lookup_unique.get(t)

            chunk['meta'] = chunk.apply(match_shop, axis=1)
            chunk = chunk.dropna(subset=['meta'])

            if chunk.empty: continue

            meta_df = pd.DataFrame(chunk['meta'].tolist(), index=chunk.index)

            chunk['official_start'] = meta_df['start_date']
            chunk['distance_to_hub'] = meta_df['distance_to_hub']
            chunk['static_zip'] = meta_df['zip_clean']
            chunk['avg_peer_dist'] = meta_df.get('avg_peer_dist')
            chunk['log_votes'] = meta_df.get('log_votes')
            chunk['State'] = meta_df.get('State')
            chunk['Category_Location'] = meta_df.get('Category_Location')

            date_col = 'date' if 'date' in chunk.columns else 'timestamp'
            chunk[date_col] = pd.to_datetime(chunk[date_col], errors='coerce')
            chunk['review_year'] = chunk[date_col].dt.year

            chunk['rating'] = pd.to_numeric(chunk['rating'], errors='coerce')
            chunk['shop_id'] = chunk['title_clean'] + "_" + chunk['static_zip'].astype(str)

            agg = chunk.groupby(['shop_id', 'review_year']).agg({
                'rating': ['sum', 'count'],
                'static_zip': 'first',
                'distance_to_hub': 'first',
                'official_start': 'first',
                'avg_peer_dist': 'first',
                'log_votes': 'first',
                'State': 'first',
                'Category_Location': 'first'
            })
            agg.columns = ['sum_rating', 'count_reviews', 'zip_clean', 'distance_to_hub', 'official_start',
                           'avg_peer_dist', 'log_votes', 'State', 'Category_Location']
            agg = agg.reset_index()
            shop_year_stats.append(agg)

        if not shop_year_stats:
            print("No matches found.")
        else:
            full_panel = pd.concat(shop_year_stats, ignore_index=True)
            print(f"Total clinic with year: {len(full_panel)}")

            final_panel = full_panel.groupby(['shop_id', 'review_year']).agg({
                'sum_rating': 'sum', 'count_reviews': 'sum', 'zip_clean': 'first',
                'distance_to_hub': 'first', 'official_start': 'first',
                'avg_peer_dist': 'first', 'log_votes': 'first', 'State': 'first', 'Category_Location': 'first'
            }).reset_index()


            final_panel['yearly_rating'] = final_panel['sum_rating'] / final_panel['count_reviews']
            min_years = final_panel.groupby('shop_id')['review_year'].min().reset_index(name='proxy_start')
            final_panel = pd.merge(final_panel, min_years, on='shop_id', how='left')
            final_panel['official_start'] = pd.to_datetime(final_panel['official_start'], errors='coerce')
            final_panel['final_start'] = final_panel['official_start'].dt.year.fillna(final_panel['proxy_start'])
            final_panel['clinic_age'] = final_panel['review_year'] - final_panel['final_start']
            final_panel = final_panel[final_panel['clinic_age'] >= 0]
            density = final_panel.groupby(['zip_clean', 'review_year'])['shop_id'].nunique().reset_index(name='active_clinics')
            final_panel = pd.merge(final_panel, density, on=['zip_clean', 'review_year'], how='left')
            density['next'] = density['review_year'] + 1
            final_panel = pd.merge(final_panel, density[['zip_clean', 'next', 'active_clinics']],
                                   left_on=['zip_clean', 'review_year'], right_on=['zip_clean', 'next'], how='left')
            final_panel.rename(columns={'active_clinics_y': 'lagged_density'}, inplace=True)
            final_panel['lagged_density'] = final_panel['lagged_density'].fillna(final_panel['active_clinics_x'])

            latest_shops = final_panel.sort_values('review_year').groupby('shop_id').tail(1)
            print(f"Unique clinic: {len(latest_shops)}")

            reg_cols = ['yearly_rating', 'clinic_age', 'lagged_density', 'distance_to_hub',
                        'avg_peer_dist', 'log_votes', 'State', 'Category_Location']
            reg_data = latest_shops.dropna(subset=reg_cols)
            print(f"Multi title: {len(reg_data)}")

            def run_quantreg(name, q):
                print(f"\n{name} (q={q})")
                try:
                    mod = smf.quantreg("yearly_rating ~ clinic_age + I(clinic_age**2) + lagged_density + distance_to_hub + avg_peer_dist + log_votes + C(State) + C(Category_Location)",
                                       data=reg_data)
                    res = mod.fit(q=q, max_iter=2000)


                    pr2 = np.nan
                    try: pr2 = res.prsquared
                    except: pass

                    adj_pr2 = np.nan
                    if res.df_resid > 0:
                        adj_pr2 = 1 - (1 - pr2) * (res.nobs - 1) / res.df_resid

                    print(f"Adj. R-squared: {adj_pr2:.4f}")
                    print(res.summary().tables[1])
                except Exception as e:
                    print(f"Error: {e}")

            run_quantreg("Quantile Regression: Low Performers", 0.1)
            run_quantreg("Quantile Regression: Median Performers", 0.5)


    except Exception as e:
        print(f"Error: {e}")